# Tips 15：画像を一括リサイズ

**業務の困りごと**：掲載画像の大きさを毎回手作業で合わせる。

**このTipsでできること**：`thumbnail((400, 400))` は縦横比を保ったまま指定範囲に収まる大きさへ縮小します。元画像とは別フォルダへ保存します。

**使用ライブラリ**：Pillow

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。各Tipsのサンプルはノートブック内で作ります。


## 1. 練習用データと準備

1200×800ピクセルの練習用画像を3枚作ります。

最後の `display()` で練習用データ全体を表として表示します。列名・行数・値を確認してから次へ進みましょう。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外の環境向け
from PIL import Image,ImageDraw
from pathlib import Path
base=Path('/content') if Path('/content').exists() else Path.cwd()
# 元画像を置くフォルダ
folder=base/'tips15_元画像'; folder.mkdir(exist_ok=True)
# 大きさ1200×800の練習用画像を3枚作る
for i,color in enumerate(['red','blue','green'],1):
    im=Image.new('RGB',(1200,800),color); im.save(folder/f'商品{i}.png')
import pandas as pd
print('サイズ変更前の画像（先頭5件）:')
display(pd.DataFrame([{'ファイル名':p.name,'画像サイズ':Image.open(p).size} for p in sorted(folder.glob('*.png'))]))


## 2. 自動処理

`thumbnail((400, 400))` は縦横比を保ったまま指定範囲に収まる大きさへ縮小します。元画像とは別フォルダへ保存します。

**結果を見るポイント**：3枚とも長辺が **400ピクセル以下** です。出力画像はZIPでまとめて保存できます。

保存した成果物を読み直して `display()` で表示します。画面の表とダウンロードしたファイルの内容を照らし合わせてください。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外の環境向け
# 縮小後の画像は別フォルダにする
outdir=base/'tips15_縮小画像'; outdir.mkdir(exist_ok=True)
# 元画像を1枚ずつ開く
for src in sorted(folder.glob('*.png')):
    with Image.open(src) as im:
        # 縦横比を保ったまま最大400×400に収める
        im.thumbnail((400,400))
        im.save(outdir/src.name,optimize=True)
print([(p.name,Image.open(p).size) for p in outdir.glob('*.png')])
# 複数の成果物をPCでまとめて扱えるようZIPにする
import shutil
out = base / 'Tips15_成果物.zip'
shutil.make_archive(str(out.with_suffix('')), 'zip', root_dir=outdir)
print('ダウンロード用ZIP:', out)

# 出力画像のサイズを表で確認する
display(pd.DataFrame([{'ファイル名': p.name, '画像サイズ': Image.open(p).size}
                      for p in sorted(outdir.glob('*.png'))]))


## 3. 結果の確認

`assert` は条件が正しいか確認する命令です。条件が違えばそこで止まるため、想定外の集計を見つけられます。

3枚とも長辺が **400ピクセル以下** です。出力画像はZIPでまとめて保存できます。


In [ ]:
# 件数や合計が期待値と一致するか検査する
assert len(list(outdir.glob('*.png')))==3
assert all(max(Image.open(p).size)<=400 for p in outdir.glob('*.png'))
print('確認OK')
assert out.exists()  # ダウンロード用ZIPも確認する


## 4. 持ち帰り・練習

**練習問題**：最大サイズを600×600に変更しましょう。

thumbnailは縦横比を保ち、元より大きく拡大しません。

**実務で使う前に**：元の小さい画像を無理に拡大しないのが `thumbnail` の特徴です。

複数のファイルをZIPでPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
# Colab上で作った成果物をPCへダウンロードする
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
